<a href="https://colab.research.google.com/github/locbp-uzh/biopipelines/blob/main/ExamplePipelines/notebooks/kinase_LID_redesign_results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kinase LID Redesign

**BioPipelines example** — de novo backbone design of the adenylate kinase LID domain using RFdiffusion, followed by ProteinMPNN sequence design and AlphaFold2 refolding. Candidates are filtered by RMSD of the fixed regions and pLDDT.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [1]:
# @title
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

Cloning into 'biopipelines'...
remote: Enumerating objects: 7105, done.
remote: Counting objects: 100% (276/276), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 7105 (delta 189), reused 187 (delta 147), pack-reused 6829 (from 3)
Receiving objects: 100% (7105/7105), 14.52 MiB | 25.49 MiB/s, done.
Resolving deltas: 100% (5274/5274), done.
/content/biopipelines
Obtaining file:///content/biopipelines
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.5 MB/s eta 0:00:00
  Building editable for biopipelines (pyproject.toml) ... done
  Created wheel for biopipelines: filename=biopipelines-1.1.0-0.editable

In [2]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.pymol import PyMOL

with Pipeline(project="Setup", job="InstallTools"):
    RFdiffusion.install()
    ProteinMPNN.install()
    AlphaFold.install()
    PyMOL.install()

Streaming output truncated to the last 5000 lines.
279950K .......... .......... .......... .......... .......... 59%  350M 4s
280000K .......... .......... .......... .......... .......... 59% 65.5M 4s
280050K .......... .......... .......... .......... .......... 59% 21.9M 4s
280100K .......... .......... .......... .......... .......... 59%  340M 4s
280150K .......... .......... .......... .......... .......... 59% 33.8M 4s
280200K .......... .......... .......... .......... .......... 59% 28.6M 4s
280250K .......... .......... .......... .......... .......... 59% 31.5M 4s
280300K .......... .......... .......... .......... .......... 59% 47.2M 4s
280350K .......... .......... .......... .......... .......... 59%  170M 4s
280400K .......... .......... .......... .......... .......... 59% 14.9M 4s
280450K .......... .......... .......... .......... .......... 59%  334M 4s
280500K .......... .......... .......... .......... .......... 59%  316M 4s
280550K .......... .......... .......

## Cell 3: Kinase LID Redesign Pipeline

Takes adenylate kinase (PDB: 4AKE) and redesigns the LID domain (residues 118–160, replaced with a new 50–70 residue loop).
The top 3 designs with RMSD < 1.5 Å on the fixed scaffold and highest pLDDT are selected.

In [3]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.rfdiffusion import RFdiffusion
from biopipelines.protein_mpnn import ProteinMPNN
from biopipelines.alphafold import AlphaFold
from biopipelines.conformational_change import ConformationalChange
from biopipelines.panda import Panda
from biopipelines.pymol import PyMOL

with Pipeline(project="AdenylateKinase", job="LID_Redesign"):
    kinase = PDB("4AKE")
    backbones = RFdiffusion(pdb=kinase,
                            contigs='A1-117/50-70/A161-214',
                            num_designs=2)
    sequences = ProteinMPNN(structures=backbones,
                            num_sequences=2,
                            redesigned=backbones.tables.structures.designed)
    refolded = AlphaFold(proteins=sequences)
    conf_change = ConformationalChange(reference_structures=kinase,
                                       target_structures=refolded,
                                       selection=backbones.tables.structures.fixed)
    top3 = Panda(tables=[refolded.tables.confidence,
                          conf_change.tables.changes],
                 operations=[Panda.merge(),
                              Panda.filter("RMSD < 1.5"),
                              Panda.sort("plddt", ascending=False),
                              Panda.head(3)],
                 pool=refolded)
top3

  4AKE not found locally, will download from RCSB

Running PDB (step 1)
=== Activating Environment ===
Requested: biopipelines
Environment: biopipelines
Location: /root/.local/share/mamba/envs/biopipelines
Python: /root/.local/share/mamba/envs/biopipelines/bin/python
Python version: Python 3.12.13
Fetching 1 structures
Convert: keep as-is (pdb|cif)
PDB IDs: 4AKE
Custom IDs: 4AKE
Priority: PDBs/ -> RCSB download
Output folder: /content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/001_PDB
Fetching 1 structures (keep as-is (pdb|cif))
Priority: PDBs/ -> RCSB download
Water molecules will be removed

[1/1] Processing 4AKE -> 4AKE
4AKE not found locally, downloading from RCSB
Cached to PDBs/ folder: /content/biopipelines/PDBs/4AKE.pdb
Successfully downloaded 4AKE as 4AKE: 297432 bytes (rcsb_download (PDB))

Successful fetches saved: /content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/001_PDB/structures.csv (1 structures)
Sequences saved: /content/biopipelines/tests/AdenylateK

StandardizedOutput({'structures': DataStream(name='structures', format='pdb', items=3, files=3, map_table=set), 'msas': DataStream(name='msas', format='a3m', items=3, files=3, map_table=set), 'rendering_parameters': {'structures': {'color_by': 'plddt', 'plddt_upper': 100}}, 'tables': {'result': TableInfo(name='result', path='/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda/merge_filter_sort.csv', columns=['id', 'structure', 'plddt', 'max_pae', 'ptm'], count=variable), 'missing': TableInfo(name='missing', path='/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda/missing.csv', columns=['id', 'removed_by', 'cause'], count=variable), 'structures': TableInfo(name='structures', path='/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda/structures_map.csv', columns=['id', 'file'], count=1), 'confidence': TableInfo(name='confidence', path='/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda/confidence.csv', columns=['id', 'structure', 'plddt', 'max_pae', 'ptm'], count=1), 'msas': TableInfo(name='msas', path='/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda/msas.csv', columns=['id', 'sequences.id', 'sequence', 'msa_file'], count=1)}, 'output_folder': '/content/biopipelines/tests/AdenylateKinase/LID_Redesign_001/006_Panda'})